In [1]:
# BLOQUE 1 - Cargar entorno, imports, helpers de presentación y mostrar estado
from dotenv import load_dotenv
import os
import logging
from bson import ObjectId
import traceback
from IPython.display import display, Markdown

# Cargar .env
load_dotenv()

# Logging mínimo
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("prueba_dao_mongo")

# Helpers de presentación para Jupyter
def md(msg):
    display(Markdown(msg))

def info(msg):
    md(f"**ℹ️ {msg}**")

def ok(msg):
    md(f"**✅ {msg}**")

def warn(msg):
    md(f"**⚠️ {msg}**")

# Mostrar estado inmediato para que la celda tenga salida
md("### Estado inicial del entorno")
info(f"MONGO_HOST = `{os.getenv('MONGO_HOST', 'no definido')}`")
info(f"MONGO_PORT = `{os.getenv('MONGO_PORT', 'no definido')}`")
info(f"MONGO_USER presente = `{bool(os.getenv('MONGO_USER'))}`")
info(f"MONGO_PASSWORD presente = `{bool(os.getenv('MONGO_PASSWORD'))}`")
info(f"Logging nivel = `{logging.getLevelName(logger.getEffectiveLevel())}`")

ok("Bloque 1 cargado correctamente. Ahora pegá y ejecutá el Bloque 2 para inicializar el DAO.")


### Estado inicial del entorno

**ℹ️ MONGO_HOST = `localhost`**

**ℹ️ MONGO_PORT = `27017`**

**ℹ️ MONGO_USER presente = `True`**

**ℹ️ MONGO_PASSWORD presente = `True`**

**ℹ️ Logging nivel = `INFO`**

**✅ Bloque 1 cargado correctamente. Ahora pegá y ejecutá el Bloque 2 para inicializar el DAO.**

In [2]:
# BLOQUE 2 - Importar e inicializar DAO (muestra estado)
try:
    from dao.usuario_dao import UsuarioDAO
    info("Módulo dao.usuario_dao importado correctamente.")
except Exception as e:
    warn(f"No se pudo importar UsuarioDAO: {e}")
    raise

def init_dao():
    """Inicializa UsuarioDAO y muestra resultado en Jupyter."""
    try:
        dao = UsuarioDAO()
        ok("DAO inicializado correctamente. Conexión establecida.")
        # Mostrar modo de conexión aproximado
        mongo_user = bool(os.getenv("MONGO_USER"))
        mongo_pwd = bool(os.getenv("MONGO_PASSWORD"))
        if mongo_user and mongo_pwd:
            info("Variables de entorno de Mongo detectadas (MONGO_USER/MONGO_PASSWORD).")
        else:
            info("No se detectaron credenciales en el entorno; se usará conexión sin auth.")
        return {"ok": True, "dao": dao}
    except Exception as e:
        warn(f"Error inicializando DAO: {e}")
        return {"ok": False, "error": str(e), "trace": traceback.format_exc()}

# Uso inmediato (ejecutar estas dos líneas tras pegar el bloque)
# res = init_dao()
# if res["ok"]: dao = res["dao"]


**ℹ️ Módulo dao.usuario_dao importado correctamente.**

In [5]:
# BLOQUE 3 - Lecturas (count, list, show) con salida inmediata
from IPython.display import display, Markdown

def md(msg):
    display(Markdown(msg))

def info(msg):
    md(f"**ℹ️ {msg}**")

def ok(msg):
    md(f"**✅ {msg}**")

def warn(msg):
    md(f"**⚠️ {msg}**")

def count_docs(dao):
    try:
        c = dao.collection.count_documents({})
        ok(f"Total documentos en `usuario`: **{c}**")
        return {"ok": True, "count": int(c)}
    except Exception as e:
        warn(f"Error al contar documentos: {e}")
        return {"ok": False, "error": str(e)}

def list_docs(dao, limit=10):
    try:
        md(f"### 🔎 Listando hasta {limit} documentos de `usuario`")
        cursor = dao.collection.find().limit(limit)
        docs = []
        for d in cursor:
            d["_id"] = str(d["_id"])
            if "municipio_id" in d and d["municipio_id"] is not None:
                d["municipio_id"] = str(d["municipio_id"])
            docs.append(d)
        if not docs:
            info("La colección está vacía.")
        else:
            for i, doc in enumerate(docs, start=1):
                md(
                    f"**Documento {i}**  \n"
                    f"- `_id`: `{doc['_id']}`  \n"
                    f"- `nombre_apellido`: `{doc.get('nombre_apellido')}`  \n"
                    f"- `dni`: `{doc.get('dni')}`  \n"
                    f"- `email`: `{doc.get('email')}`"
                )
        return {"ok": True, "docs": docs}
    except Exception as e:
        warn(f"Error al listar documentos: {e}")
        return {"ok": False, "error": str(e)}

def show_doc(dao, id_str):
    try:
        md(f"### 📄 Buscando documento `_id` = `{id_str}`")
        doc = dao.collection.find_one({"_id": ObjectId(id_str)})
        if not doc:
            warn("Documento no encontrado.")
            return {"ok": False, "error": "not_found"}
        doc["_id"] = str(doc["_id"])
        if "municipio_id" in doc and doc["municipio_id"] is not None:
            doc["municipio_id"] = str(doc["municipio_id"])
        md("**Documento encontrado:**")
        for k, v in doc.items():
            md(f"- **{k}**: `{v}`")
        return {"ok": True, "doc": doc}
    except Exception as e:
        warn(f"Error al buscar documento: {e}")
        return {"ok": False, "error": str(e)}



In [6]:
res = init_dao()
if not res["ok"]:
    print("Error init:", res)
else:
    dao = res["dao"]


INFO:dao.usuario_dao:Conectado a Mongo sin autenticación (fallback).


**✅ DAO inicializado correctamente. Conexión establecida.**

**ℹ️ Variables de entorno de Mongo detectadas (MONGO_USER/MONGO_PASSWORD).**

In [7]:
count_docs(dao)


**✅ Total documentos en `usuario`: **14****

{'ok': True, 'count': 14}

In [8]:
out = list_docs(dao, limit=5)
out  # para inspeccionar el dict retornado


### 🔎 Listando hasta 5 documentos de `usuario`

**Documento 1**  
- `_id`: `6a20b3b3064ffad2f013b0e1`  
- `nombre_apellido`: `Juan Perez`  
- `dni`: `30111222`  
- `email`: `juan.perez@chilecito.com`

**Documento 2**  
- `_id`: `6a20b3b3064ffad2f013b0e2`  
- `nombre_apellido`: `Maria Gomez`  
- `dni`: `31111222`  
- `email`: `maria.g@chilecito.com`

**Documento 3**  
- `_id`: `6a20b3b3064ffad2f013b0e3`  
- `nombre_apellido`: `Carlos Ruiz`  
- `dni`: `32111222`  
- `email`: `carlos.r@chilecito.com`

**Documento 4**  
- `_id`: `6a20b3b3064ffad2f013b0e4`  
- `nombre_apellido`: `Ana Torres`  
- `dni`: `33111222`  
- `email`: `ana.t@chilecito.com`

**Documento 5**  
- `_id`: `6a20b3b3064ffad2f013b0e5`  
- `nombre_apellido`: `Luis Diaz`  
- `dni`: `34111222`  
- `email`: `luis.d@larioja.com`

{'ok': True,
 'docs': [{'_id': '6a20b3b3064ffad2f013b0e1',
   'nombre_apellido': 'Juan Perez',
   'dni': '30111222',
   'reputacion': 50,
   'email': 'juan.perez@chilecito.com',
   'contrasena': 'pwd123',
   'municipio_id': '6a20b3b3064ffad2f013b0df'},
  {'_id': '6a20b3b3064ffad2f013b0e2',
   'nombre_apellido': 'Maria Gomez',
   'dni': '31111222',
   'reputacion': 45,
   'email': 'maria.g@chilecito.com',
   'contrasena': 'pwd123',
   'municipio_id': '6a20b3b3064ffad2f013b0df'},
  {'_id': '6a20b3b3064ffad2f013b0e3',
   'nombre_apellido': 'Carlos Ruiz',
   'dni': '32111222',
   'reputacion': 60,
   'email': 'carlos.r@chilecito.com',
   'contrasena': 'pwd123',
   'municipio_id': '6a20b3b3064ffad2f013b0df'},
  {'_id': '6a20b3b3064ffad2f013b0e4',
   'nombre_apellido': 'Ana Torres',
   'dni': '33111222',
   'reputacion': 55,
   'email': 'ana.t@chilecito.com',
   'contrasena': 'pwd123',
   'municipio_id': '6a20b3b3064ffad2f013b0df'},
  {'_id': '6a20b3b3064ffad2f013b0e5',
   'nombre_apellido':

In [9]:
show_doc(dao, "6a20ead36712d192d5cf686d")


### 📄 Buscando documento `_id` = `6a20ead36712d192d5cf686d`

**Documento encontrado:**

- **_id**: `6a20ead36712d192d5cf686d`

- **nombre_apellido**: `Prueba Usuario`

- **dni**: `77777777`

- **reputacion**: `10`

- **email**: `prueba.usuario@example.com`

- **contrasena**: `pwd_demo`

- **municipio_id**: `6a20b3b3064ffad2f013b0df`

{'ok': True,
 'doc': {'_id': '6a20ead36712d192d5cf686d',
  'nombre_apellido': 'Prueba Usuario',
  'dni': '77777777',
  'reputacion': 10,
  'email': 'prueba.usuario@example.com',
  'contrasena': 'pwd_demo',
  'municipio_id': '6a20b3b3064ffad2f013b0df'}}

In [10]:
# DEMO SEGURO: ejecutar solo lecturas por defecto
if __name__ == "__main__":
    res = init_dao()
    if not res.get("ok"):
        print("Error inicializando DAO:", res.get("error"))
    else:
        dao = res["dao"]
        print("Total documentos:", count_docs(dao).get("count"))
        out = list_docs(dao, limit=3)
        print("Primeros documentos (resumen):", [d["_id"] for d in out.get("docs", [])])
        print("Demo finalizado. No se realizaron escrituras.")


INFO:dao.usuario_dao:Conectado a Mongo sin autenticación (fallback).


**✅ DAO inicializado correctamente. Conexión establecida.**

**ℹ️ Variables de entorno de Mongo detectadas (MONGO_USER/MONGO_PASSWORD).**

**✅ Total documentos en `usuario`: **14****

Total documentos: 14


### 🔎 Listando hasta 3 documentos de `usuario`

**Documento 1**  
- `_id`: `6a20b3b3064ffad2f013b0e1`  
- `nombre_apellido`: `Juan Perez`  
- `dni`: `30111222`  
- `email`: `juan.perez@chilecito.com`

**Documento 2**  
- `_id`: `6a20b3b3064ffad2f013b0e2`  
- `nombre_apellido`: `Maria Gomez`  
- `dni`: `31111222`  
- `email`: `maria.g@chilecito.com`

**Documento 3**  
- `_id`: `6a20b3b3064ffad2f013b0e3`  
- `nombre_apellido`: `Carlos Ruiz`  
- `dni`: `32111222`  
- `email`: `carlos.r@chilecito.com`

Primeros documentos (resumen): ['6a20b3b3064ffad2f013b0e1', '6a20b3b3064ffad2f013b0e2', '6a20b3b3064ffad2f013b0e3']
Demo finalizado. No se realizaron escrituras.
